In [ ]:

import pandas as pd
from pathlib import Path

project_dir = Path(
    "C:/Users/markg/OneDrive/Desktop/DACCS/DACSS 756 - Machine Learning/DACSS_ML_FP"
)


def MarksCSVRead(target_file_name):
    # Check that the project directory actually exists
    if not project_dir.exists():
        raise FileNotFoundError(
            f"Project directory does not exist:\n{project_dir}"
        )

    if not project_dir.is_dir():
        raise NotADirectoryError(
            f"This path is not a directory:\n{project_dir}"
        )

    print(f"Searching in:\n{project_dir}")
    print()

    # Recursively search all folders
    for file_path in project_dir.rglob(target_file_name):
        print(f"Found file: {file_path}")
        return pd.read_csv(file_path)

    raise FileNotFoundError(
        f"Could not find '{target_file_name}' anywhere under:\n{project_dir}"
    )






In [ ]:
con = MarksCSVRead("connections_princeton.csv.gz")

print(con.columns.tolist())
print(f"rows: {len(con):,}")
print(f"min syn_count: {con['syn_count'].min()}")  # must be >= 5 otherwise, need to filter

Searching in:
C:\Users\markg\OneDrive\Desktop\DACCS\DACSS 756 - Machine Learning\DACSS_ML_FP

Found file: C:\Users\markg\OneDrive\Desktop\DACCS\DACSS 756 - Machine Learning\DACSS_ML_FP\Flywire_FAFBv783\connections_princeton.csv.gz
['pre_root_id', 'post_root_id', 'neuropil', 'syn_count', 'nt_type']
rows: 5,342,446
min syn_count: 1


In [ ]:
import pandas as pd

cls = MarksCSVRead("classification.csv.gz")
cts = MarksCSVRead("consolidated_cell_types.csv.gz")
con = MarksCSVRead("connections_princeton.csv.gz")

print("cell types columns:", cts.columns.tolist())   # check before trusting below

# --- join type labels onto the classification table
meta = cls.merge(cts, on="root_id", how="left")
type_col = "primary_type" if "primary_type" in meta.columns else "cell_type"
print(f"using type column: {type_col}")
print(f"neurons with a type: {meta[type_col].notna().sum():,} of {len(meta):,}")

# --- edges: aggregate (already thresholded at >=5 per pair)
edges = (con.groupby(["pre_root_id", "post_root_id"], as_index=False)["syn_count"]
            .sum())

# --- neurotransmitter per neuron (already per-neuron; just dedupe)
nt = con[["pre_root_id", "nt_type"]].drop_duplicates("pre_root_id") \
        .rename(columns={"pre_root_id": "root_id"})
meta = meta.merge(nt, on="root_id", how="left")

# --- sign convention: glutamate INHIBITORY
SIGN = {"ACH": +1, "GLUT": -1, "GABA": -1,
        "SER": 0, "DA": 0, "OCT": 0}        # modulators excluded from W
meta["sign"] = meta["nt_type"].map(SIGN).fillna(0)
meta["is_modulatory"] = meta["nt_type"].isin(["SER", "DA", "OCT"])

# --- mushroom body selection
mb_pat = r"^KC|^MBON|^PAM|^PPL1|^PPL2|^APL"
mb = meta[meta[type_col].astype(str).str.contains(mb_pat, case=False,
                                                  na=False, regex=True)]
print(f"\nMB neurons: {len(mb):,}")
print(mb[type_col].str.extract(r"^([A-Za-z']+)")[0].value_counts().head(12))
print(f"\nby side:\n{mb['side'].value_counts()}")
print(f"modulatory (your Tier 3 reward channel): {mb['is_modulatory'].sum():,}")

# --- the MB subgraph
ids = set(mb["root_id"])
sub = edges[edges["pre_root_id"].isin(ids) & edges["post_root_id"].isin(ids)]
print(f"\nMB subgraph: {len(ids):,} neurons, {len(sub):,} edges")
print(f"mean degree: {2*len(sub)/max(len(ids),1):.1f}")
print(f"density: {len(sub)/max(len(ids)*(len(ids)-1),1):.4f}")

Searching in:
C:\Users\markg\OneDrive\Desktop\DACCS\DACSS 756 - Machine Learning\DACSS_ML_FP

Found file: C:\Users\markg\OneDrive\Desktop\DACCS\DACSS 756 - Machine Learning\DACSS_ML_FP\Flywire_FAFBv783\classification.csv.gz
Searching in:
C:\Users\markg\OneDrive\Desktop\DACCS\DACSS 756 - Machine Learning\DACSS_ML_FP

Found file: C:\Users\markg\OneDrive\Desktop\DACCS\DACSS 756 - Machine Learning\DACSS_ML_FP\Flywire_FAFBv783\consolidated_cell_types.csv.gz
Searching in:
C:\Users\markg\OneDrive\Desktop\DACCS\DACSS 756 - Machine Learning\DACSS_ML_FP

Found file: C:\Users\markg\OneDrive\Desktop\DACCS\DACSS 756 - Machine Learning\DACSS_ML_FP\Flywire_FAFBv783\connections_princeton.csv.gz
cell types columns: ['root_id', 'primary_type', 'additional_type(s)']
using type column: primary_type
neurons with a type: 138,327 of 139,255

MB neurons: 5,606
0
KCg       2489
KCab      1772
KCapbp     916
PAM        307
MBON        96
PPL         24
APL          2
Name: count, dtype: int64

by side:
side
rig

In [ ]:
import numpy as np, scipy.sparse as sp
from scipy.sparse.csgraph import connected_components

ids_sorted = sorted(ids)
idx = {r: i for i, r in enumerate(ids_sorted)}
n = len(idx)

sign_map = dict(zip(meta["root_id"], meta["sign"]))
r = sub["pre_root_id"].map(idx).to_numpy()
c = sub["post_root_id"].map(idx).to_numpy()
w = sub["syn_count"].to_numpy(float) * sub["pre_root_id"].map(sign_map).to_numpy(float)

A = sp.csr_matrix((w, (r, c)), shape=(n, n))

# 1. how much of the graph is actually recurrent?
ncomp, lab = connected_components(A, directed=True, connection="strong")
sizes = np.bincount(lab)
print(f"strongly connected components: {ncomp:,}")
print(f"largest SCC: {sizes.max():,} neurons ({sizes.max()/n:.1%} of subgraph)")

# 2. spectral radius of the signed matrix
ev = sp.linalg.eigs(A.astype(float), k=1, which="LM",
                    return_eigenvectors=False, maxiter=5000)
print(f"spectral radius (raw weights): {abs(ev[0]):.3f}")

# 3. where do the edges actually go?
grp = dict(zip(meta["root_id"],
               meta[type_col].astype(str).str.extract(r"^([A-Za-z']+)")[0]))
flow = (sub.assign(pre=sub["pre_root_id"].map(grp),
                   post=sub["post_root_id"].map(grp))
           .groupby(["pre", "post"])["syn_count"]
           .agg(["count", "sum"])
           .sort_values("count", ascending=False))
print(flow.head(20))

strongly connected components: 71
largest SCC: 5,535 neurons (98.7% of subgraph)
spectral radius (raw weights): 1522.683
               count    sum
pre    post                
KCab   KCab    11590  74385
KCg    MBON    10414  82528
KCab   MBON    10081  80137
KCapbp MBON     6442  64267
KCg    KCg      6332  38612
KCapbp KCapbp   6217  37672
APL    KCg      2489  62498
KCg    APL      2489  66440
APL    KCab     1772  50628
KCab   APL      1772  50062
KCapbp PPL      1375   9634
KCab   PPL      1178   7231
KCapbp PAM      1150   7075
KCg    PPL      1092   6408
PPL    KCg      1085   6382
KCapbp APL       918  33147
APL    KCapbp    917  33895
PPL    KCapbp    811   5185
KCab   PAM       777   5310
PAM    MBON      758  11806
